# DataModel NetCDF Example

This notebook shows how to save and load a custom `DataModel` to NetCDF with nested and mixed array payloads.

In [1]:
from pathlib import Path

import numpy as np
import tunits

from qubex.core.model import DataModel

In [2]:
class DemoResult(DataModel):
    """Demo model with nested arrays and tunits values."""

    name: str
    waveform: np.ndarray
    delay: tunits.ValueArray
    iq_blocks: dict[str, np.ndarray]
    traces: list[np.ndarray]
    metadata: dict[str, str]

In [3]:
result = DemoResult(
    name="demo",
    waveform=np.array([[1 + 2j, 3 + 4j], [5 + 6j, 7 + 8j]]),
    delay=tunits.ValueArray([10, 20], tunits.ns),
    iq_blocks={
        "readout": np.array([0.1 + 0.2j, 0.2 + 0.3j]),
        "reference": np.array([0.0 + 0.0j, 0.5 + 0.1j]),
    },
    traces=[
        np.array([1.0, 2.0, 3.0], dtype=np.float32),
        np.array([1, 2, 3, 4], dtype=np.int32),
    ],
    metadata={"author": "qubex", "experiment": "demo"},
)
print(result)

name='demo' waveform=array([[1.+2.j, 3.+4.j],
       [5.+6.j, 7.+8.j]]) delay=ValueArray(array([10, 20]), 'ns') iq_blocks={'readout': array([0.1+0.2j, 0.2+0.3j]), 'reference': array([0. +0.j , 0.5+0.1j])} traces=[array([1., 2., 3.], dtype=float32), array([1, 2, 3, 4], dtype=int32)] metadata={'author': 'qubex', 'experiment': 'demo'}


In [4]:
output = Path(".rawdata/demo_data_model.nc")
output.parent.mkdir(parents=True, exist_ok=True)
saved = result.save_netcdf(output)
print(saved)

.rawdata/demo_data_model.nc


In [5]:
restored = DemoResult.load_netcdf(saved)
print("name:", restored.name)
print("waveform:", restored.waveform.shape, restored.waveform.dtype)
print("delay:", restored.delay, restored.delay.units)
print("iq_blocks:", {k: (v.shape, v.dtype) for k, v in restored.iq_blocks.items()})
print("trace dtypes:", [t.dtype for t in restored.traces])
print("metadata:", restored.metadata)

name: demo
waveform: (2, 2) complex128
delay: [10. 20.] ns ns
iq_blocks: {'readout': ((2,), dtype('complex128')), 'reference': ((2,), dtype('complex128'))}
trace dtypes: [dtype('float64'), dtype('int64')]
metadata: {'author': 'qubex', 'experiment': 'demo'}


## NetCDF4 tutorial: attributes, dimensions, variables

- **Global attributes**: file-level metadata (`format`, `model_class`, ...)
- **Dimensions**: named axis definitions (`dim_0`, ...)
- **Variables**: actual arrays that reference dimensions


In [ ]:
from netCDF4 import Dataset

with Dataset(saved, mode="r", auto_complex=True) as ds:
    print("[global attributes]")
    for key in ds.ncattrs():
        value = ds.getncattr(key)
        print(f"- {key}: {value}")

    print("\n[dimensions]")
    for name, dim in ds.dimensions.items():
        print(f"- {name}: size={len(dim)}, unlimited={dim.isunlimited()}")

    print("\n[variables]")
    for name, var in ds.variables.items():
        print(f"- {name}: dims={var.dimensions}, shape={var.shape}, dtype={var.dtype}")

[global attributes]
- format: qxdata
- format_version: 1
- model_class: __main__.DemoResult
- payload_json: {"name": "demo", "waveform": {"__variable__": {"name": "waveform", "type": "numpy.ndarray", "shape": [2, 2], "units": []}}, "delay": {"__variable__": {"name": "delay", "type": "tunits.ValueArray", "shape": [2], "units": [{"unit": "SECOND", "scale": "NANO"}]}}, "iq_blocks": {"readout": {"__variable__": {"name": "iq_blocks_readout", "type": "numpy.ndarray", "shape": [2], "units": []}}, "reference": {"__variable__": {"name": "iq_blocks_reference", "type": "numpy.ndarray", "shape": [2], "units": []}}}, "traces": [{"__variable__": {"name": "traces_i0", "type": "numpy.ndarray", "shape": [3], "units": []}}, {"__variable__": {"name": "traces_i1", "type": "numpy.ndarray", "shape": [4], "units": []}}], "metadata": {"author": "qubex", "experiment": "demo"}, "__meta__": {"format": "qxdata", "version": 1}}

[dimensions]
- dim_waveform_0: size=2
- dim_waveform_1: size=2
- dim_delay_0: size=2
-

## Inspect payload structure

`payload_json` stores the model structure, while array values live in NetCDF variables.


In [7]:
import json

with Dataset(saved, mode="r", auto_complex=True) as ds:
    payload = ds.getncattr("payload_json")
    obj = json.loads(payload)

print("top-level keys:", list(obj.keys()))
print("waveform ref:", obj["waveform"])
print("first trace ref:", obj["traces"][0])

top-level keys: ['name', 'waveform', 'delay', 'iq_blocks', 'traces', 'metadata', '__meta__']
waveform ref: {'__variable__': {'name': 'waveform', 'type': 'numpy.ndarray', 'shape': [2, 2], 'units': []}}
first trace ref: {'__variable__': {'name': 'traces_i0', 'type': 'numpy.ndarray', 'shape': [3], 'units': []}}
